# Parallel and distributed image processing

The **short-tailed shearwater**, otherwise known as the **yolla** or **muttonbird** (*Ardenna tenuirostris*) is the most abundant seabird in Australian waters. It breeds mainly on small islands in Bass Strait and Tasmania, and migrates to the Northern Hemisphere for the boreal summer. Breeding is done in burrows in sandy soil, where chicks are raised until the departure of the parents, and the fledglings begin their own migration.

![Short-tailed shearwater near Burrow, Bruny Island, Tasmania, Australia (https://www.jjharrison.com.au/)](extras/Puffinus_tenuirostris_Bruny.jpg)

The traditional method of counting numbers of shearwater chicks is to create transects across the known breeding grounds, and to manually traverse checking for chicks. Unfortunately this can have a negative impact on the colonies - frightening the adult birds and in the worst case collapsing burrows in the unstable earth.

An alternative is to use infra-red imagery captured using drones mounted with FLIR (forward-looking infra-red) cameras taken at night. Hotspots in this infra-red imagery can correspond to occupied burrows so using this imagery can be a low impact method of capturing counts. Hotspots can of course be counted manually, but you can use image processing tools from the [scikit-image](https://scikit-image.org/) and the [dask-image](https://image.dask.org/) libraries to capture these hotspots in an automated process.

This notebook is a parallel version of our earlier chapter / notebook "Image processing". We use the same example of finding bird nests in infra-red imagery, but this time, instead of using serial image processing routines from *scikit-image*, we use *dask-image* to handle larger datasets in parallel.

Thanks to Jacob Virtue (University of Tasmania) for providing the imagery.

### Exercise: dask-image coverage

The serial version of this notebook used the following functions:

- maximum filter
- Sobel filter
- binary dilation

Which of these are available in dask-image? Check on this page:

http://image.dask.org/en/latest/coverage.html

What else is implemented that you could use in your work?

## Imports

In [1]:
# Dask Image is a library for parallel image processing
# Install with: uv pip install dask-image
import dask.array as da
import dask_image as di
import dask_image.imread
from dask_image import ndfilters, ndmorph, ndmeasure

# And some other libraries for plotting and data manipulation
import numpy as np
import matplotlib.pyplot as plt
from dask.distributed import Client

# Initialize Dask client for distributed computing
client = Client()  # This sets up local distributed computing

## Reading and masking the data

The bird nest imagery is captured as a single-band TIFF image (i.e. a greyscale image measuring intensity).
Previously we read these as a NumPy array.
Now we will instead read these as a Dask array using the `imread` function in *dask-image*:

This data contains values of 65535 to represent invalid data (outside the boundary of interest).
Dask supports masked arrays like NumPy.
When working with large image data, we often encounter invalid or missing values that need special handling.
Dask provides masked arrays similar to NumPy's masked arrays, allowing us to handle these cases efficiently.

In [12]:
def load_and_mask_image(
    filepath: str, 
    na_value: float = 65535.0,
    chunks: tuple[int, int] | str | None = None
) -> da.Array:
    """
    Load and mask an image using dask-image, handling invalid values.
    
    Args:
        filepath: Path to the image file
        na_value: Value representing invalid data
        chunks: Chunk size for dask array. Can be:
            - tuple[int, int] for explicit chunk sizes per dimension
            - 'auto' for automatic chunking
            - None to use the default chunking (default)
        
    Returns:
        Masked dask array with invalid values handled
    """
    # Load image
    img = di.imread.imread(filepath)[0, ...]
    
    # Create mask for invalid values
    mask = (img == na_value)
    
    # Create masked array
    img_uint8 = img.astype(np.uint8)

    # Rechunk if chunks specified. Note you have to do this after the conversion
    # otherwise dask doesn't propagate the mask correctly (bug?)
    if chunks is not None:
        img_uint8 = img_uint8.rechunk(chunks)
    
    return da.ma.masked_array(img_uint8, mask=mask, fill_value=0)

# Example usage:
breeding_grounds = load_and_mask_image(
    '/Data/LargeData/breeding_grounds_large.tif',
    chunks=(1000, 1000)  # Creates 1000x1000 pixel chunks
)

# Inspect chunking
print("Array shape:", breeding_grounds.shape)
print("Chunk structure:", breeding_grounds.chunks)
breeding_grounds

Masked arrays contain a mask that is True or False to indicate invalid pixels and they help us in several ways:
1. They hide the invalid values from computations, so they don't affect the results.
2. Operations on masked arrays are automatically masked, so they don't propagate invalid values.
3. Dask's masked arrays are lazy, so they don't compute the mask until necessary, i.e. when we call `.compute()`.

#+INCLUDE: "attachments/bc167114ab252625bb8b2d8743cb8474.md"

We can retrieve the masked array information and work with masked arrays similarily to NumPy arrays:

In [14]:
# Get mask array
mask = da.ma.getmaskarray(breeding_grounds)

# Count valid pixels
valid_pixels = da.count_nonzero(~mask).compute()

# Basic statistics on valid data only
mean_value = da.ma.mean(breeding_grounds).compute()

If you load the data with chunks, you'll see that the chunks are 1000x1000 pixels in size.
To analyse these see the following function:

In [15]:
def analyze_chunks(dask_array: da.Array) -> None:
    """
    Analyze and display chunk information for a dask array.
    
    Args:
        dask_array: Dask array to analyze
    """
    # Basic chunk information
    print("Array shape:", dask_array.shape)
    print("Chunk structure:", dask_array.chunks)
    
    # Calculate number of chunks
    num_chunks = len(dask_array.chunks[0]) * len(dask_array.chunks[1])
    print(f"Total number of chunks: {num_chunks}")
    
    # Estimate memory per chunk (assuming uint8)
    chunk_size = dask_array.chunks[0][0] * dask_array.chunks[1][0]
    chunk_mb = chunk_size / (1024 * 1024)  # Convert to MB
    print(f"Approximate size per chunk: {chunk_mb:.2f} MB")
    
    # Visualize chunk structure
    print("\nChunk layout (each number represents one chunk):")
    chunk_grid = np.arange(num_chunks).reshape(
        len(dask_array.chunks[0]), 
        len(dask_array.chunks[1])
    )
    print(chunk_grid)

This data is captured at approximately 3cm resolution, so this section of imagery covers roughly 2.3 hectares of ground:

In [19]:
import pint

u = pint.UnitRegistry()

area = da.count_nonzero(~ da.isnan(breeding_grounds)).compute() * (9 * u.cm**2)
round(area.to(u.hectare), 2)

#+INCLUDE: "attachments/432893d8ee4658ad52b76492e4b885ec.md"

It's always a good idea to visualise this data as well.
Matplotlib automatically handles masked arrays, so we can plot the data directly:

In [18]:
import matplotlib.pyplot as plt

plt.subplots(figsize=(10, 10))
plt.imshow(breeding_grounds, cmap='Greys');

#+INCLUDE: "attachments/e75a16e2a518a33c227939846b23f4ad.md"

Notice the small dark marks -- these are hotspots which are occupied shearwater burrows (with the entrances being approximately 15cm to 20cm across).

Although you can see the hotspots in this region easily, the camera is an adaptive camera, so hotspots in different areas will have different values -- thus a simple thresholding of the data will not work.
A visualisation of a histogram of the cell values may still be useful.


When working with both masked data and large datasets, ensure that you try not to call `.compute()` too early, as this will load the entire dataset into memory.
Instead, use Dask's lazy evaluation to perform operations on the data without loading it into memory.
Further, you can alter the chunk size to optimize the performance of your computations, specifically RAM usage and computation speed.
Finally, for testing purposes, it is often useful to work with a subset of the data before applying the operations to the entire dataset.

Another visualisation we can do is to plot a histogram of the pixel values in the image.
This will tell us the distribution of pixel values in the image, which can be useful for setting thresholds or understanding the data better.
While scikit-image has a `skimage.exposure.histogram` function, *dask-image* does not wrap this function. 
However, dask does have a `histogram` method that we can use.
Here is a small wrapper around it to work with a masked array:

In [20]:
def compute_histogram(
    dask_array: da.Array, bins: int = 256, normalize: bool = True
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute histogram of a dask array, optionally normalizing the results.

    Args:
        dask_array: Input dask array (can be masked)
        bins: Number of bins for histogram
        normalize: Whether to normalize the histogram

    Returns:
        tuple of (histogram counts, bin centers)
    """
    # Get only valid values from masked array
    valid_data = dask_array[~da.ma.getmaskarray(dask_array)]

    # Compute min and max for bin ranges
    data_min = da.min(valid_data)
    data_max = da.max(valid_data)

    # Create histogram using dask
    hist, bin_edges = da.histogram(
        valid_data, bins=bins, range=(data_min, data_max))

    # We need to compute() here to get actual values
    hist, bin_edges = hist.compute(), bin_edges.compute()

    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # Normalize if requested
    if normalize:
        hist = hist / hist.sum()

    return hist, bin_centers

plt.figure(figsize=(10, 6))
hist, centers = compute_histogram(breeding_grounds, bins=256, normalize=True)
plt.plot(centers, hist)
plt.title('Normalized Histogram of Valid Pixel Values')
plt.xlabel('Pixel Value')
plt.ylabel('Normalized Frequency')
plt.grid(True, alpha=0.3)
plt.show()

#+INCLUDE: "attachments/dc78f84a415a1d13085010e1f3a65204.md"

### Exercises

Using the `breeding_grounds` masked array:

1. Calculate the percentage of invalid (masked) pixels
2. Find the mean value of valid pixels
3. Create a boolean mask showing where values are above the mean

Note: Remember to use `.compute()` when needed!

### Extended Exercise

Research and implement a way to estimate the memory usage of the `breeding_grounds` array:
1. Calculate the theoretical memory usage of the full array
2. Calculate the actual memory usage of the chunked dask array
3. Compare the two and explain the difference

Hint: Look into `sys.getsizeof()` and dask array memory management

For solutions, see *solutions/dask_image_memory_usage.py*.


## Highlighting thermal hotspots using rank filters

The burrows where the shearwaters live are quite small.
They can be challenging to detected in the raw data due to noise and the adaptive nature of the camera.

To help your processes pick out these regions it may be useful to expand these contiguous regions.  This can be done with a rank filtering. Rank filters can be used for several purposes such as:

- image quality enhancement e.g. image smoothing, sharpening
- image pre-processing e.g. noise reduction, contrast enhancement
- feature extraction e.g. border detection, isolated point detection
- post-processing e.g. small object removal, object grouping, contour smoothing

In this case you will use a maximum filter to remove noise and increase the contrast.
Maximum filtering preserves the bright regions in the image, while removing the dark regions and suppressing the noise.


A maximum filter operates by sliding a window over the image and replacing the center pixel with the maximum value in the window.
It is implemented as a rank filter in scikit-image, and a parallel version is available in dask-image.
The footprint parameter specifies the shape of the window, which can be a square, circle, or any other shape.
Values in this window are multiplied by the footprint, and the maximum value is returned.

In this example this expands the burrows to make them easier to detect.
Further, it preserves the peak temperature of the burrow.

In the serial version of this image processing pipeline (in the chapter on "Image processing"), we used a maximum filter from `skimage.filters`. While *dask-image* does not wrap this exactly, *dask-image* contains a parallel version of the `maximum_filter` function in `scipy.ndimage`.

Notice first how the `maximum_filter` function in `scipy.ndimage` is equivalent to the maximum rank-filter available in scikit-image:

In [22]:
test_region = breeding_grounds[2600:2800, 2800:3000]

from scipy.ndimage import maximum_filter
from skimage.filters import rank
from skimage.morphology import disk


footprint = disk(3)

import time
results = {}
times = {}

# scikit-image
start = time.time()
results['skimage'] = rank.maximum(test_region, footprint=footprint)
times['skimage_time'] = time.time() - start

# scipy
start = time.time()
results['scipy'] = maximum_filter(test_region, footprint=footprint)
times['scipy_time'] = time.time() - start

# dask-image
start = time.time()
results['dask'] = di.ndfilters.maximum_filter(
    image[region], 
    footprint=footprint
).compute()
times['dask_time'] = time.time() - start

### Exercises
1. Are the results the same? Hint: use `np.all()` or `da.all()` and `.compute()`
2. How long does each version take? Which is faster: serial or parallel?
3. Try the same analysis on the entire dataset
4. Write a function that takes a dask array and a footprint and compares the results before and after the maximum filter

Hint: use `plt.subplots()` to plot the results side by side

In [23]:
fig, axes = plt.subplots(1, 2, figsize=(12, 12))
axes[0].imshow(results['skimage'], cmap='Greys')
axes[0].set_title('scikit-image')
axes[1].imshow(results['scipy'], cmap='Greys')
axes[1].set_title('scipy.ndimage')

### Extended Exercises
1. Compare the maximum filter with these other filters:
- Gaussian filter (`dif.gaussian_filter`)
- Median filter (`dif.median_filter`)
- Sobel filter (`dif.sobel`)

2. Use `skimage.metrics.peak_signal_noise_ratio` and `skimage.metrics.structural_similarity` to compare the results of the filters. What do these metrics tell you about the filters?

For solutions, see *solutions/dask_image_filters.py*.

## Edge detection in thermal images

After enhancing the thermal hotspots with maximum filtering, we need to identify the boundaries of potential nests.
Edge detection is crucial because:

1. Nest boundaries show sharp temperature transitions
1. Edge patterns help distinguish nests from other thermal anomalies
1. Clear boundaries aid in measuring nest sizes and separating nearby nests

In this case you will use a Sobel filter which will give you the magnitude of the edges between pixels.
The Sobel operator detects edges by measuring intensity gradients in two directions:

1. Vertical changes (north-south temperature differences)
1. Horizontal changes (east-west temperature differences)
1. Combined to give overall edge strength

Previously we used the `sobel` function in `skimage.filters`. A near-equivalent in *dask-image* is `dask_image.ndfilters.sobel`, although we require a second step: to take the absolute value to get an overall magnitude once the Sobel filter has been processed.

In [41]:
from skimage.filters import sobel as sk_sobel
from scipy.ndimage import sobel as nd_sobel
from dask_image.ndfilters import sobel as di_sobel

Once again we'll examine a small region to see the effects before applying the filter more broadly.

In [42]:
small_region = ranked_breeding_grounds[2600:2800, 2800:3000]

#### Preprocessing

The first step performed by *scikit-image*'s `sobel` function is to convert the
image to float64 and scale the values from 0 to 1. The final step is to compute
the absolute value of the filter results. We perform these steps manually
before and after applying the `sobel` function from `dask_image.ndfilters` (or
from `scipy.ndimage`) to achieve the same result:

In [43]:
def scale_to_range(
    arr: da.Array,
    out_min: float = 0.0,
    out_max: float = 1.0
) -> da.Array:
    """
    Scale array values to specified range.
    
    Args:
        arr: Input array
        out_min: Minimum value of output range
        out_max: Maximum value of output range
        
    Returns:
        Scaled dask array
    """
    arr_min = da.min(arr)
    arr_max = da.max(arr)
    
    # Avoid division by zero
    scale = (arr_max - arr_min) or 1.0
    
    scaled = (arr - arr_min) / scale
    return scaled * (out_max - out_min) + out_min


def detect_edges(
    image: da.Array,
    scale_range: bool = True,
    threshold_percentile: float | None = None
) -> da.Array:
    """
    Detect edges in thermal imagery using Sobel filters.
    
    Args:
        image: Input dask array
        scale_range: Whether to scale input to [0,1] range
        threshold_percentile: Optional percentile for edge thresholding
        
    Returns:
        Edge magnitude image as dask array
    """
    # Scale image if requested
    if scale_range:
        image = scale_to_range(image)
    
    # Compute gradients
    grad_y = dif.sobel(image, axis=0)
    grad_x = dif.sobel(image, axis=1)
    
    # Compute magnitude
    magnitude = da.sqrt(grad_x**2 + grad_y**2)
    
    # Threshold if requested
    if threshold_percentile is not None:
        threshold = da.percentile(magnitude, threshold_percentile)
        return da.where(magnitude >= threshold, 1, 0)
    
    return magnitude

Next we can analyze the effects of the Sobel filter on the small region:

In [44]:
small_region = ranked_breeding_grounds[2600:2800, 2800:3000]

# Step 1: Enhance thermal hotspots with a maximum filter
enhanced = dif.maximum_filter(breeding_grounds, footprint=disk(3))

# Step 2: Detect edges on enhanced image
edges = detect_edges(enhanced, scale_range=True)

# Step 3: Threshold edges
# Flatten array for percentile computation
edges_flat = edges.ravel()
threshold = da.percentile(edges_flat, 99).compute()
edges_thresholded = da.where(edges >= threshold, 1, 0)
    
# Visualize a region
region = (slice(2600, 2800), slice(2800, 3000))
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.ravel()

axes[0].imshow(breeding_grounds[region].compute(), cmap='Greys')
axes[0].set_title('Original')

axes[1].imshow(enhanced[region].compute(), cmap='Greys')
axes[1].set_title('After Maximum Filter')

axes[2].imshow(edges[region].compute(), cmap='Greys')
axes[2].set_title('Edge Magnitude')

axes[3].imshow(edges_thresholded[region].compute(), cmap='Greys')
axes[3].set_title('Thresholded Edges')

plt.tight_layout()

### Exercise
1. Apply a thresholding to remove weak edges. Use the 99th percentile as the threshold. Hint: use `da.percentile()`, but it doesn't accept multiple dimensions. You may need to flatten the array first. Pass the threshold to `da.where()` to threshold the edges.
1. Create a function that runs the pipeline to this point. The function should take a dask image array and visualise the difference between the starting image and final image.

### Extended Exercise

1. Use the enhanced edges to:
    1. Measure nest diameters
    2. Identify connected nests
    3. Calculate nest density
    4. Validate against known nest sizes (15-20cm)
2. Create an adaptive thresholding system that:
    1. Analyzes local image statistics
    2. Adjusts threshold based on local contrast
    3. Handles varying background temperatures
    4. Works efficiently with dask arrays
    Hint: Research: Look into local histogram analysis techniques

For solutions, see *solutions/dask_image_edge_detection.py*.

## Refining nest boundaries with morphological operations

There is still noise in the results though.
Ideally you want to remove small "salt and pepper" and thin edges which make up noise in the data, while at the same time keeping the burrows distinct.
You can do this through morphology operations, which allow us to connect broken edges, remove noise, and separate overlapping objects.

The two basic operations are `dilation` and `erosion` (with their equivalents in dask-image `binary_dilation` and `binary_erosion` in the `ndmorph` module)
Dilation expands the region by a structuring element, while erosion reduces the region by a structuring element.

In [65]:
from dask_image import ndmorph

small_region = edges_thresholded[2600:2800, 2800:3000]

fig, plots = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))
plots[0, 0].imshow(
    small_region,
    cmap='Greys'
)
plots[0, 0].set_title('Thresholded edges');
plots[0, 1].imshow(
    ndmorph.binary_dilation(small_region, structure=disk(3)),
    cmap='Greys'
)
plots[0, 1].set_title('Dilation');
plots[1, 0].imshow(
    ndmorph.binary_erosion(small_region, structure=disk(3)),
    cmap='Greys'
)
plots[1, 0].set_title('Erosion');
plots[1, 1].imshow(
    ndmorph.binary_erosion(ndmorph.binary_dilation(small_region, structure=disk(3)), structure=disk(3)),
    cmap='Greys'
)
plots[1, 1].set_title('Dilation + Erosion');

#+INCLUDE: "attachments/55afa104fae801f648f25738d2877fe7.md"

Dilation expands a region by some structuring element, erosion reduces the region by a structuring element. Since a dilation will fill a hole in a region, following it with an erosion will not open the holes again. This common operation is known as a `closing` operation, and there is a shortcut function for this in dask-image as the `binary_closing` function.

If you do this for a slightly larger disk than your original filter region (which gave the nest edges) you should close any nest regions:

In [69]:
closed_breeding_grounds = ndmorph.binary_closing(
    threshold_breeding_grounds,
    structure=disk(3)
)
plt.subplots(figsize=(10, 10))
plt.imshow(closed_breeding_grounds[2600:2800, 2800:3000], cmap='Greys');

#+INCLUDE: "attachments/865cec1dd4025fa22f4ace8e636a95f5.md"

This still leaves thin lines. You can follow the closing operation with an `opening` operation -- which is a shortcut for an erosion followed by a dilation. This means all the areas will shrink (and thin lines will dissapear), before the larger areas are expanded back to their original size. In this case you'll open by the same sized disk as the original maximum filter:

In [71]:
opened_breeding_grounds = ndmorph.binary_opening(
    closed_breeding_grounds,
    structure=disk(3)
)
plt.subplots(figsize=(10, 10))
plt.imshow(opened_breeding_grounds[2600:2800, 2800:3000], cmap='Greys');

#+INCLUDE: "attachments/afc64d1f161bacbf91d2805daddbf896.md"

In [81]:
plt.figure(figsize=(10,10))
plt.imshow(opened_breeding_grounds, cmap='Greys')

#+INCLUDE: "attachments/3e379fb5822cdfdf1785e49cbb4cd639.md"


### Exercises

1. Implement a function that takes a dask array and a disk size and performs the above operations of closing and then opening. Ensure that the structure is a parameter that can be passed to the function.
1. Apply the closing and opening operations to the entire dataset. Visualize the results.
1. Compare different sizes for the disk structure. How does the disk size affect the results?
1. Try a square structure by passing `square(3)` to the function. How does this compare to the disk structure?

### Extended Exercise

1. Try write a custom structure element that specifically looks for nests. For example, a disk with a hole in the middle. How does this affect the results?
1. Write a validation script that generates random "nests", then adds noise and temperature variations. Use the script to test the pipeline, quantify the results, and use this to optimize the parameters.

For solutions, see *solutions/dask_image_morphology.py*.

## Labelling discrete regions and outlining nests

You now have candidates for the locations of the nests.
The next step is to decide whether these might contain a nest or not by examining each site individually.  
To do this you must uniquely label each region.

In [78]:
from dask_image import ndmeasure

small_region = opened_breeding_grounds[2600:2800, 2800:3000]
labels, num_features = ndmeasure.label(small_region)

Next we can visualise the results.

In [79]:
def visualize_labels(labels: da.Array):
    """
    Visualize labeled regions.
    
    Args:
        labels: Labeled array from ndmeasure.label
        region: Optional region to view as (row_slice, col_slice)
    """
    plt.figure(figsize=(10, 5))
    plt.imshow(labels.compute(), cmap='nipy_spectral')
    plt.title(f'Labeled Regions')
    plt.colorbar()
    plt.show()


visualize_labels(labels)

#+INCLUDE: "attachments/ff405cce470aea8d29d35d5db2be714a.md"

Once that is done, we can compute some properties about the given region. 
We need to use a delayed function for this, so that dask can parallelize the computation.

In [80]:
from dask import delayed

@delayed
def basic_region_properties(data):
    """Basic properties for a region"""
    if data is None or len(data) == 0:
        return None
        
    return {
        'area': np.sum(data),
        'mean_val': np.mean(data),
        'std_val': np.std(data)
    }

def analyze_basic_properties(
    binary_image: da.Array,
    labels: da.Array,
    num_features: int
) -> list:
    """Get basic properties for all regions"""
    props = ndmeasure.labeled_comprehension(
        binary_image,
        labels,
        index=da.arange(1, num_features + 1),
        func=basic_region_properties,
        out_dtype=object,
        default=None
    )
    
    return props.compute()

# We can then compute the properties with:
analyze_basic_properties(opened_breeding_grounds, labels, num_features)

The properties here are in pixels, so we need to next move to real-world units.

In [82]:
@delayed
def nest_properties(data, pixel_size_cm: float = 3.0):
    """Compute nest-specific properties"""
    if data is None or len(data) == 0:
        return None
        
    area_pixels = np.sum(data)
    area_cm2 = area_pixels * pixel_size_cm**2
    diameter_cm = 2 * np.sqrt(area_pixels / np.pi) * pixel_size_cm
    
    # Filter based on expected nest size (15-20cm diameter)
    if not (15 <= diameter_cm <= 20):
        return None
        
    return {
        'area_cm2': area_cm2,
        'diameter_cm': diameter_cm
    }

def analyze_nest_sizes(
    binary_image: da.Array,
    labels: da.Array,
    num_features: int
) -> tuple[list, int]:
    """Analyze regions and filter by size"""
    props = ndmeasure.labeled_comprehension(
        binary_image,
        labels,
        index=da.arange(1, num_features + 1),
        func=nest_properties,
        out_dtype=object,
        default=None
    )
    
    properties = props.compute()
    valid_nests = [p for p in properties if p is not None]
    
    print(f"Total regions: {num_features}")
    print(f"Valid nest candidates: {len(valid_nests)}")
    return valid_nests, len(valid_nests)

valid_nests, num_nests = analyze_nest_sizes(opened_breeding_grounds, labels, num_features)

### Exercise

1. Using the original image, calculate the mean temperature of the valid nests (remember that the original image has pixel intensities for temperature).
2. Tie everything together into a single function that takes the original image, processes it using the techniques shown so far, and returns the number of valid nests and their mean temperature. Utilize the functions you have written so far.

For solutions, see *solutions/dask_image_nest_detection.py*.

### Extended Exercise

1. Write a function that generates synthetic nest data. This should create random "nests" of varying sizes and temperatures. Use this to test the pipeline and optimize the parameters.
1. Overlay the nest boundaries on the original image. How well do the boundaries match the actual nests?
1. Try writing a better `regionprops` function to extract the nests.

Hint: Given what we know of the shearwater birds, the nest is around 20cm in diameter (or 7 cells) and they can nest quite closely together. So assuming a 7 cell diameter nest, which may have been joined to a neighbouring nest (or two) through the opening and closing methodology, we will approximate the maximum area of a candidate region to be 75 cells (this is arbitrary, and should be considered on a case by case basis). We can use the `ndmeasure.area` algorithm to find the area of each region and filter out those we don't care about.

For solutions, see *solutions/dask_image_nest_detection_extended.py*.